In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
import os

In [ ]:
result_total = pd.read_csv('../data/results_df_spatial_vessel_linkerenz.csv', index_col = 0)

In [ ]:
import statsmodels.api as sm
import statsmodels.formula.api as smf

results_df = result_total.copy()

# 2) GEE
#   cov_struct=sm.cov_struct.Exchangeable() : assumes observations within the same cluster share a common correlation structure
#   other structures such as Independence or AR(1) can also be tried as needed
model_gee = smf.gee(
    "dist_cancer_area ~ C(clinical_type)",
    groups="sample_id",               # cluster (group) variable
    data=results_df,                   
    family=sm.families.Gaussian(),    
    cov_struct=sm.cov_struct.Exchangeable()
)
results_gee = model_gee.fit()

print(results_gee.summary())

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# p-value -> significance stars
p_value = results_gee.pvalues[1]
if p_value < 0.001:
    stars = '***'
elif p_value < 0.01:
    stars = '**'
elif p_value < 0.05:
    stars = '*'
else:
    stars = 'ns'

# Split the data by group
resistant = results_df.loc[results_df['clinical_type'] == 'Resistant', 'dist_cancer_area']
sensitive = results_df.loc[results_df['clinical_type'] == 'Sensitive', 'dist_cancer_area']
res_pre   = results_df.loc[(results_df['clinical_type'] == 'Resistant') & (results_df['time_point'] == 'pre'),  'dist_cancer_area']
res_post  = results_df.loc[(results_df['clinical_type'] == 'Resistant') & (results_df['time_point'] == 'post'), 'dist_cancer_area']

# Common settings (bracket position)
jitter = 0.04
y_max = max(resistant.max(), sensitive.max())
y_min = min(resistant.min(), sensitive.min())
h = (y_max - y_min) * 0.05 if y_max > y_min else 0.05
y = y_max + h


# ── 2) Violin Plot ─────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 6), dpi=300)

vp = ax.violinplot(
    [sensitive, resistant],             # left = Sensitive, right = Resistant
    positions=[1, 2],
    widths=0.6,
    showextrema=False
)
colors = ['#2ca02c', '#1f77b4']         # left = green (Sensitive), right = blue (Resistant)
for body, col in zip(vp['bodies'], colors):
    body.set_facecolor(col)
    body.set_edgecolor('black')
    body.set_alpha(0.3)
    body.set_linewidth(2)

# Scatter points
ax.scatter(np.random.normal(2, jitter, size=len(res_pre)),  res_pre,   s=50, alpha=0.8, c='#1f77b4', marker='o', label='Resistant (pre)')
ax.scatter(np.random.normal(2, jitter, size=len(res_post)), res_post,  s=50, alpha=0.8, c='#ff7f0e', marker='s', label='Resistant (post)')
ax.scatter(np.random.normal(1, jitter, size=len(sensitive)), sensitive, s=50, alpha=0.8, c='#2ca02c', marker='D', label='Sensitive')

# p-value bracket / text
ax.plot([1, 1, 2, 2], [y, y + h, y + h, y], lw=1.5, c='black')
ax.text(1.5, y + h * 1.2, f"{stars}  (p = {p_value:.3f})", ha='center', va='bottom', fontsize=10)

# Style / labels
ax.set_axisbelow(True)
for spine in ['top', 'right']:
    ax.spines[spine].set_visible(False)
ax.set_xticks([1, 2])
ax.set_xticklabels(['Sensitive', 'Resistant'], fontsize=12)
ax.set_xlabel("Sensitivity to T-Dxd", fontsize=15, labelpad = 16)
ax.set_ylabel("Distance between Cancer and Vessel (µm)", fontsize=15, labelpad = 15)
ax.set_title('Total samples', fontsize = 20, y = 1.1)


# Legend: Sensitive on top
handles, labels = ax.get_legend_handles_labels()
label_to_idx = {lab: i for i, lab in enumerate(labels)}
order = [label_to_idx[k] for k in ['Sensitive', 'Resistant (pre)', 'Resistant (post)'] if k in label_to_idx]
ax.legend([handles[i] for i in order], [labels[i] for i in order],
          frameon=False, fontsize=14, loc='upper left', bbox_to_anchor=(1.02, 1))

fig.tight_layout(rect=[0, 0, 0.85, 1])
plt.show()


In [ ]:
results_df_pos = results_df[results_df['HER2_type'] == 'Pos'].copy()

model_gee = smf.gee(
    "dist_cancer_area ~ C(clinical_type)",
    groups="sample_id",               # cluster (group) variable
    data=results_df_pos,                   
    family=sm.families.Gaussian(),    
    cov_struct=sm.cov_struct.Exchangeable()
)
results_gee = model_gee.fit()

print(results_gee.summary())

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Example p-value (replace with your precomputed value)
p_value = results_gee.pvalues[1]

# Significance-star string
if p_value < 0.001:
    stars = '***'
elif p_value < 0.01:
    stars = '**'
elif p_value < 0.05:
    stars = '*'
else:
    stars = 'ns'

# Split the data by group
resistant = results_df_pos.loc[results_df_pos['clinical_type'] == 'Resistant', 'dist_cancer_area']
sensitive = results_df_pos.loc[results_df_pos['clinical_type'] == 'Sensitive', 'dist_cancer_area']
res_pre   = results_df_pos.loc[(results_df_pos['clinical_type'] == 'Resistant') & (results_df_pos['time_point'] == 'pre'),  'dist_cancer_area']
res_post  = results_df_pos.loc[(results_df_pos['clinical_type'] == 'Resistant') & (results_df_pos['time_point'] == 'post'), 'dist_cancer_area']

# Common settings
jitter = 0.04
y_max = max(resistant.max(), sensitive.max())
y_min = min(resistant.min(), sensitive.min())
h = (y_max - y_min) * 0.05 if y_max > y_min else 0.05
y = y_max + h

# --- 2) Violin Plot + light interior fill + p-value number annotation ---
fig, ax = plt.subplots(figsize=(8, 6), dpi=300)

vp = ax.violinplot(
    [sensitive, resistant],             # left = Sensitive, right = Resistant
    positions=[1, 2],
    widths=0.6,
    showextrema=False
)
colors = ['#2ca02c', '#1f77b4']         # left (Sensitive) = green, right (Resistant) = blue
for body, col in zip(vp['bodies'], colors):
    body.set_facecolor(col)
    body.set_edgecolor('black')
    body.set_alpha(0.3)
    body.set_linewidth(2)

ax.scatter(np.random.normal(2, jitter, size=len(res_pre)),  res_pre,   s=50, alpha=0.8, c='#1f77b4', marker='o', label='Resistant (pre)')   # x=2
ax.scatter(np.random.normal(2, jitter, size=len(res_post)), res_post,  s=50, alpha=0.8, c='#ff7f0e', marker='s', label='Resistant (post)') # x=2
ax.scatter(np.random.normal(1, jitter, size=len(sensitive)), sensitive, s=50, alpha=0.8, c='#2ca02c', marker='D', label='Sensitive')        # x=1

# p-value line and text
ax.plot([1, 1, 2, 2], [y, y + h, y + h, y], lw=1.5, c='black')
ax.text(1.5, y + h * 1.2, f"{stars}  (p = {p_value:.3f})", ha='center', va='bottom', fontsize=10)

# Style / labels
ax.set_axisbelow(True)
for spine in ['top', 'right']:
    ax.spines[spine].set_visible(False)
ax.set_xticks([1, 2])
ax.set_xticklabels(['Sensitive', 'Resistant'], fontsize=12)
ax.set_xlabel("Sensitivity to T-Dxd", fontsize=15, labelpad = 16)
ax.set_ylabel("Distance between Cancer and Vessel (µm)", fontsize=15, labelpad = 15)
ax.set_title('HER2 positive', fontsize = 20, y = 1.1)

# legend: Sensitive on top
handles, labels = ax.get_legend_handles_labels()
label_to_idx = {lab: i for i, lab in enumerate(labels)}
order = [label_to_idx[k] for k in ['Sensitive', 'Resistant (pre)', 'Resistant (post)'] if k in label_to_idx]
ax.legend([handles[i] for i in order], [labels[i] for i in order],
          frameon=False, fontsize=14, loc='upper left', bbox_to_anchor=(1.02, 1))

fig.tight_layout(rect=[0, 0, 0.85, 1])
plt.show()


In [ ]:
results_df_low = results_df[results_df['HER2_type'] == 'Low'].copy()

model_gee = smf.gee(
    "dist_cancer_area ~ C(clinical_type)",
    groups="sample_id",               # cluster (group) variable
    data=results_df_low,                   
    family=sm.families.Gaussian(),    
    cov_struct=sm.cov_struct.Exchangeable()
)
results_gee = model_gee.fit()

print(results_gee.summary())

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Example p-value (replace with your precomputed value)
p_value = results_gee.pvalues[1]

# Significance-star string
if p_value < 0.001:
    stars = '***'
elif p_value < 0.01:
    stars = '**'
elif p_value < 0.05:
    stars = '*'
else:
    stars = 'ns'

# Split the data by group
resistant = results_df_low.loc[results_df_low['clinical_type'] == 'Resistant', 'dist_cancer_area']
sensitive = results_df_low.loc[results_df_low['clinical_type'] == 'Sensitive', 'dist_cancer_area']
res_pre   = results_df_low.loc[(results_df_low['clinical_type'] == 'Resistant') & (results_df_low['time_point'] == 'pre'),  'dist_cancer_area']
res_post  = results_df_low.loc[(results_df_low['clinical_type'] == 'Resistant') & (results_df_low['time_point'] == 'post'), 'dist_cancer_area']

# Common settings
jitter = 0.04
y_max = max(resistant.max(), sensitive.max())
y_min = min(resistant.min(), sensitive.min())
h = (y_max - y_min) * 0.05 if y_max > y_min else 0.05
y = y_max + h


# --- 2) Violin Plot + light interior fill + p-value number annotation ---
fig, ax = plt.subplots(figsize=(8, 6), dpi=300)

vp = ax.violinplot(
    [sensitive, resistant],             # <- order swapped: [Sensitive, Resistant]
    positions=[1, 2],
    widths=0.6,
    showextrema=False
)
colors = ['#2ca02c', '#1f77b4']         # <- left (Sensitive) = green, right (Resistant) = blue
for body, col in zip(vp['bodies'], colors):
    body.set_facecolor(col)
    body.set_edgecolor('black')
    body.set_alpha(0.3)
    body.set_linewidth(2)

ax.scatter(np.random.normal(2, jitter, size=len(res_pre)),  res_pre,   s=50, alpha=0.8, c='#1f77b4', marker='o', label='Resistant (pre)')  # x=2
ax.scatter(np.random.normal(2, jitter, size=len(res_post)), res_post,  s=50, alpha=0.8, c='#ff7f0e', marker='s', label='Resistant (post)') # x=2
ax.scatter(np.random.normal(1, jitter, size=len(sensitive)), sensitive, s=50, alpha=0.8, c='#2ca02c', marker='D', label='Sensitive')       # x=1

# p-value line and text
ax.plot([1, 1, 2, 2], [y, y + h, y + h, y], lw=1.5, c='black')
ax.text(1.5, y + h * 1.2, f"{stars}  (p = {p_value:.3f})", ha='center', va='bottom', fontsize=10)

# Style / labels
ax.set_axisbelow(True)
for spine in ['top', 'right']:
    ax.spines[spine].set_visible(False)
ax.set_xticks([1, 2])
ax.set_xticklabels(['Sensitive', 'Resistant'], fontsize=12)
ax.set_xlabel("Sensitivity to T-Dxd", fontsize=15, labelpad = 16)
ax.set_ylabel("Distance between Cancer and Vessel (µm)", fontsize=15, labelpad = 15)
ax.set_title('HER2 low', fontsize = 20, y = 1.1)
ax.set_ylim(0, 200)

# legend: Sensitive on top
handles, labels = ax.get_legend_handles_labels()
label_to_idx = {lab: i for i, lab in enumerate(labels)}
order = [label_to_idx[k] for k in ['Sensitive', 'Resistant (pre)', 'Resistant (post)'] if k in label_to_idx]
ax.legend([handles[i] for i in order], [labels[i] for i in order],
          frameon=False, fontsize=14, loc='upper left', bbox_to_anchor=(1.02, 1))

fig.tight_layout(rect=[0, 0, 0.85, 1])
plt.show()
